### 3. Qual é o tempo médio de duração total de um processo de Falência desde a decretação até o encerramento da fase de arrecadação e pagamento de credores?

**Pergunta:** decretada a falência, os credores devem receber sua parte da
massa falida até o processo se encerrar. A pergunta mede esse trecho final:
da decretação ao encerramento.

**Fonte no mart (consumo puro):** `gold_base_analitica` — a duração já vem
calculada em `dias_duracao_falencia` (Bloco 4 do notebook 03): da data da
decretação da falência até o encerramento do processo; quando o
encerramento não é registrado, usa-se a extinção como desfecho alternativo.
Este notebook não refaz esses cálculos, mas consome o resultado pronto.

**Recorte:** classe 108 (falência originária), com decretação e desfecho
registrados e duração coerente (≥ 0). Agregação Brasil, estratificada por
regime legal — `escopo_lei_atual` separa o que foi ajuizado a partir de
09/06/2005 (Lei 11.101/2005) do que é anterior (Lei 7.661/1945, regime
antigo que ainda existe no corpus).

**Limitações:**
1. O "fim da fase de arrecadação e pagamento" não é um movimento nomeado
   na TPU — o marco adotado é o encerramento/extinção do processo, que na
   prática sinaliza o fecho da massa;
2. Processos decretados mas sem desfecho registrado ficam fora da média
   (contados no denominador de cobertura);
3. Recuperações judiciais convoladas em falência (classe 129) ficam fora
   deste recorte;
4. Os processos do regime antigo são, em média, mais antigos e foram
   registrados em sistemas com menos consistência — a comparação entre
   regimes herda esse viés de época.

#### O que a consulta a seguir mede e como (cobertura da duração)

**Entrada:** todos os campos vêm da tabela `default.gold_base_analitica`,
filtrada para falências originárias (classe 108), sem filtro de regime —
a cobertura é medida sobre o corpus inteiro.

A consulta lê duas colunas por processo:

| Coluna lida | O que representa |
|---|---|
| `data_decretao_falencia` | a data da decretação da falência (nulo = o processo ainda não foi decretado) |
| `dias_duracao_falencia` | dias da decretação ao encerramento/extinção, já calculado na gold (nulo ou negativo = duração não mensurável) |

**Saída — o que ela produz (uma única linha com três números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_falencias` | conta todos os processos da classe 108 | o universo do recorte (15.932) |
| `com_decretacao` | conta os casos com decretação datada | quantas falências entraram efetivamente na fase falimentar (4.180) |
| `com_duracao_medida` | conta os casos com decretação e duração ≥ 0 | quantas têm o trecho decretação → encerramento mensurável (847) — o denominador efetivo do comparativo |

**Em uma frase:** a consulta mede quanto do corpus de falências
originárias tem a duração decretação → encerramento mensurável,
estabelecendo a cobertura de 20,3% que condiciona todas as leituras
seguintes.

In [0]:
%sql
-- Q3 · Cobertura: quantas falências têm a duração mensurável
SELECT
  count(*)                                                        AS total_falencias,
  count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END)  AS com_decretacao,
  count(CASE WHEN data_decretao_falencia IS NOT NULL
              AND dias_duracao_falencia >= 0 THEN 1 END)          AS com_duracao_medida
FROM default.gold_base_analitica
WHERE classe_codigo = 108;

total_falencias,com_decretacao,com_duracao_medida
15932,4180,847


#### O que a próxima consulta mede e como (comparativo entre regimes)

**Entrada:** as mesmas colunas da consulta anterior, restrita aos casos
elegíveis (decretação datada e duração ≥ 0), com o regime legal
derivado de `escopo_lei_atual`.

**Método:** estatísticas de duração (média, mediana, P25, P90) por
regime legal — Lei 7.661/1945 (anterior a 2005) vs. Lei 11.101/2005
(vigente). A mediana é a métrica central: a média fica sensível às
falências longas de cauda.

**Saída — o que ela produz (duas linhas, uma por regime, com seis números cada):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `qtd_falencias` | conta os casos elegíveis do regime | o n de cada regime (337 antiga / 510 vigente) |
| `media_dias` | média de `dias_duracao_falencia` | o tempo médio, sensível à cauda longa |
| `mediana_dias` | percentil 50 | a duração típica — a métrica recomendada |
| `p25_dias` / `p90_dias` | percentis 25 e 90 | a dispersão: onde começa a distribuição e onde ela engrossa |

**Em uma frase:** a consulta compara a duração da fase falimentar entre
os dois regimes legais do corpus, produzindo as quatro medidas que
mostram a aceleração trazida pela reforma de 2005 em toda a
distribuição — não só no caso típico.

In [0]:
%sql
-- Q3 · Comparativo de regimes legais: Lei 7.661/1945 (antiga) x Lei 11.101/2005
WITH elegiveis AS (
  SELECT numero_processo, dias_duracao_falencia,
         CASE WHEN escopo_lei_atual THEN 'Lei 11.101/2005'
              ELSE 'Lei 7.661/1945 (antiga)' END AS regime_legal
  FROM default.gold_base_analitica
  WHERE classe_codigo = 108
    AND data_decretao_falencia IS NOT NULL
    AND dias_duracao_falencia >= 0
)
SELECT
  regime_legal,
  count(*)                                                  AS qtd_falencias,
  round(avg(dias_duracao_falencia), 1)                      AS media_dias,
  round(percentile_approx(dias_duracao_falencia, 0.5), 1)   AS mediana_dias,
  round(percentile_approx(dias_duracao_falencia, 0.25), 1)  AS p25_dias,
  round(percentile_approx(dias_duracao_falencia, 0.9), 1)   AS p90_dias
FROM elegiveis
GROUP BY 1
ORDER BY qtd_falencias DESC;

regime_legal,qtd_falencias,media_dias,mediana_dias,p25_dias,p90_dias
Lei 11.101/2005,510,1284.8,831,296,3399
Lei 7.661/1945 (antiga),337,3271.1,2033,774,7815


#### Análise da resposta (pergunta 3)

**Cobertura:** existem 15.932 processos de falência (classe 108) no mart;
4.180 têm decretação registrada e, destes, 847 (20,3%) têm a duração
mensurável — o restante ainda não tem encerramento/extinção registrado,
reflexo da duração longa da fase falimentar (a mesma censura vista na
pergunta 2).

**Comparativo de regimes — duração da falência (decretação → encerramento):**

| Indicador | Lei 7.661/1945 (antiga) | Lei 11.101/2005 (vigente) | Razão antiga/vigente |
|---|---|---|---|
| Falências com duração medida | 337 | 510 | — |
| **Mediana (dias)** | **2.033 (~5,6 anos)** | **831 (~2,3 anos)** | **2,4×** |
| Média (dias) | 3.271 (~9,0 anos) | 1.285 (~3,5 anos) | 2,5× |
| P25 (dias) | 774 | 296 | 2,6× |
| P90 (dias) | 7.815 (~21,4 anos) | 3.399 (~9,3 anos) | 2,3× |

**Leitura:** a Lei 11.101/2005 encurtou a falência típica em cerca de 60% —
a mediana caiu de ~5,6 anos (regime antigo) para ~2,3 anos. O achado mais
forte está no P25: mesmo o quarto mais rápido do regime antigo (774 dias)
levou mais que o dobro do equivalente atual (296 dias) — a aceleração
atingiu toda a distribuição, não só os casos simples. A cauda também
encurtou: o P90 caiu de ~21 para ~9 anos. A média continuar bem acima da
mediana nos dois regimes (no antigo, 1,6×) mostra que as falências longas
e complexas são um traço permanente do instituto — atenuado pela reforma,
mas não eliminado.

**Ressalva metodológica:** processos do regime antigo são mais antigos e
foram registrados com menos consistência (vieses de época), e a cobertura
da duração é de 20,3% das decretadas. A direção do efeito, porém, bate
com o que a reforma de 2005 se propunha: tramitação mais célere da
insolvência.

#### Resposta final (pergunta 3)

O processo de falência, medido da decretação ao encerramento, dura
**2,3 anos na mediana** sob a lei vigente (831 dias; média de 1.285) —
menos da metade do regime anterior (5,6 anos), com aceleração uniforme
em toda a distribuição. Mesmo assim, 1 em cada 10 falências do regime vigente ultrapassa
9 anos, e a fase de arrecadação e pagamento de credores segue sendo o
trecho mais longo da insolvência. 

Ressalvas: o marco final é o
encerramento/extinção do processo (a TPU não tem movimento próprio para
o fim da fase de rateio) e a duração é mensurável em 20,3% das
falências decretadas — as demais ainda aguardam desfecho registrado.